In [ ]:
# 1. Install ZSTD dependency and Ollama
!sudo apt-get update && sudo apt-get install zstd -y
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start Ollama server in the background
import subprocess
import time

# Use subprocess to run 'ollama serve' so it doesn't block the cell
subprocess.Popen(['ollama', 'serve'])

# Wait a few seconds for the server to wake up
time.sleep(5)

# 3. Pull Llama 3
!ollama pull llama3

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,937 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restri

In [ ]:
!pip install faiss-cpu sentence-transformers pandas ragas openai datasets tqdm unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 119.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

In [ ]:
!pip install langchain_ollama

In [ ]:
!ollama pull llama3

In [ ]:
import os

# Define BASE_PATH again for clarity in this diagnostic cell
BASE_PATH = "/content/drive/MyDrive/Embedding_model_Fine-tuning"
INDEX_PATH = f"{BASE_PATH}/medical_Kb.index"

print(f"Checking for file existence at: {INDEX_PATH}")
if os.path.exists(INDEX_PATH):
    print("File found! Proceeding to next steps.")
else:
    print("File NOT found at the specified path.")
    print("Listing contents of BASE_PATH directory:")
    !ls -l "{BASE_PATH}"

Checking for file existence at: /content/drive/MyDrive/Embedding_model_Fine-tuning/medical_Kb.index
File NOT found at the specified path.
Listing contents of BASE_PATH directory:
total 19866
-rw------- 1 root root   870602 Mar  8 07:37 eval_dataset.json
-rw------- 1 root root 15360045 Mar  8 07:37 medical_kb.index
drwx------ 6 root root     4096 Mar  8 06:50 Medical_MiniLM_Tuned
drwx------ 2 root root     4096 Mar  8 05:19 output
-rw------- 1 root root  4103149 Mar  8 07:37 passages.json


In [ ]:
import os
import faiss
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from google.colab import drive
from datasets import Dataset
from unsloth import FastSentenceTransformer
from sentence_transformers import SentenceTransformer
from ragas import evaluate
from ragas.metrics import Faithfulness, ContextPrecision
from langchain_ollama import ChatOllama

# --- 1. MOUNT DRIVE & PATHS ---
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Embedding_model_Fine-tuning"
LORA_PATH = f"{BASE_PATH}/Medical_MiniLM_Tuned/final_lora_model"

# --- 2. LOAD LLM (LLAMA 3) ---
# Ensure Ollama is running and Llama3 is pulled before running this
llm = ChatOllama(model="llama3", temperature=0)

# --- 3. LOAD MODELS & DATA ---
print("Loading Assets...")
base_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
ft_model = FastSentenceTransformer.from_pretrained(LORA_PATH, for_inference=True)

with open(f"{BASE_PATH}/passages.json", "r") as f:
    passages = json.load(f)

# Load FT Index (The one you saved)
ft_index = faiss.read_index(f"{BASE_PATH}/medical_kb.index")

# --- 4. CREATE BASE INDEX (For Fair Comparison) ---
print("Creating a temporary Base Index for a fair 'Before' baseline...")
# We use a subset of passages or the whole thing depending on RAM
# Let's use the first 10,000 to match your training size
sample_passages = passages[:10000]
base_embeddings = base_model.encode(sample_passages, show_progress_bar=True)
base_index = faiss.IndexFlatL2(base_embeddings.shape[1])
base_index.add(np.array(base_embeddings).astype('float32'))

# --- 5. RAG PIPELINE FUNCTION ---
def get_rag_response(query, model, current_index):
    # Step A: Retrieve
    q_emb = model.encode([query])
    _, indices = current_index.search(np.array(q_emb).astype('float32'), k=3)
    retrieved_contexts = [passages[i] for i in indices[0]]

    # Step B: Generate with Llama 3
    context_text = "\n".join(retrieved_contexts)
    prompt = f"""Use ONLY the following context to answer the question.
If the answer isn't in the context, say 'I don't know'.

Context:
{context_text}

Question: {query}
Answer:"""

    response = llm.invoke(prompt)
    return response.content, retrieved_contexts

# --- 6. RUN COMPARISON AUDIT ---
eval_df = pd.read_json(f"{BASE_PATH}/eval_dataset.json", orient="records", lines=True).head(10)
base_results, ft_results = [], []

print(f"Auditing {len(eval_df)} samples...")
for _, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    q = row["anchor"]
    gt = row["positive"]

    # Run Base RAG (Base Model + Base Index)
    ans_b, ctx_b = get_rag_response(q, base_model, base_index)
    base_results.append({"question": q, "answer": ans_b, "contexts": ctx_b, "ground_truth": gt})

    # Run FT RAG (Fine-tuned Model + Fine-tuned Index)
    ans_f, ctx_f = get_rag_response(q, ft_model, ft_index)
    ft_results.append({"question": q, "answer": ans_f, "contexts": ctx_f, "ground_truth": gt})

# --- 7. RAGAS EVALUATION ---
print("\nCalculating RAGAS Scores...")
base_eval = evaluate(Dataset.from_list(base_results), metrics=[Faithfulness(), ContextPrecision()], llm=llm)
ft_eval = evaluate(Dataset.from_list(ft_results), metrics=[Faithfulness(), ContextPrecision()], llm=llm)

# --- 8. FINAL SUMMARY TABLE ---
# We calculate the mean (average) of the scores for a clear comparison
comparison_df = pd.DataFrame({
    "Metric": ["Faithfulness (Hallucination)", "Context Precision (Retrieval)"],
    "Base Model (Generic)": [
        np.mean(base_eval["faithfulness"]),
        np.mean(base_eval["context_precision"])
    ],
    "Fine-tuned Model (Medical)": [
        np.mean(ft_eval["faithfulness"]),
        np.mean(ft_eval["context_precision"])
    ]
})

print("\n" + "="*60)
print("FINAL PERFORMANCE COMPARISON (AVERAGED)")
print("="*60)
print(comparison_df.to_string(index=False))

# Calculate % Improvement
improvement = ((comparison_df.iloc[0, 2] - comparison_df.iloc[0, 1]) / comparison_df.iloc[0, 1]) * 100
print(f"\n🚀 Total Hallucination Reduction: {improvement:.2f}%")

/tmp/ipykernel_9157/1757904.py:12: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision
/tmp/ipykernel_9157/1757904.py:12: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Assets...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

100%|██████████| 10/10 [00:26<00:00,  2.64s/it]



Calculating RAGAS Scores...


Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/20 [00:00<?, ?it/s]


FINAL PERFORMANCE COMPARISON (AVERAGED)
                       Metric  Base Model (Generic)  Fine-tuned Model (Medical)
 Faithfulness (Hallucination)              0.563333                    0.696667
Context Precision (Retrieval)              0.816667                    0.783333

🚀 Total Hallucination Reduction: 23.67%
